# VWAP Stdev Bands

## Contents

- [Configuration](#configuration)
  - [Setup](#setup)
  - [Automatic](#automatic)
  - [Manual](#manual)
  - [Final configuration](#final-configuration)
- [VWAP Stdev Bands](#vwap-bands)
  - [Backtesting](#vwap-bands-backtesting)
  - [Grid search](#vwap-bands-grid-search)
  - [Walk-forward analysis](#vwap-bands-walk-forward)
  - [Monte Carlo simulations](#vwap-bands-monte-carlo)
  - [Live signals](#vwap-bands-live-signals)
- [VWAP Analysis](#vwap-analysis)
  - [Band hit frequency](#band-hit-frequency)
  - [Tuning the entry band](#tuning-the-entry-band)
  - [Entry-band sweep](#entry-band-sweep)

Algorithm: session-anchored VWAP with volume-weighted stdev bands → mean-reversion entry on a furthest-band breach → exit on return to the VWAP middle.

Pattern:
1. Compute a daily-anchored VWAP and five upper/lower bands at \
   (1.28, 2.01, 2.51, 3.09, 4.01) standard deviations (same defaults as the \
   TradingView "VWAP Stdev Bands v2 Mod" preset).
2. When close crosses through the furthest band (from inside → outside), enter \
   a counter-trend trade: LONG below the lower band, SHORT above the upper band.
3. Exit when the close returns to the VWAP middle.

Features:
- VWAP + stdev recomputed cumulatively within each session — bands reset every UTC day.
- Causal by construction: bar i only depends on bars 0..i of the active session, \
  so look-ahead is structurally impossible (same contract enforced by the backtester).
- All five stdev multipliers, the session anchor, and the entry-band index are \
  exposed on VwapParams — no magic numbers in the strategy file.

How the VWAP Bands Algorithm Determines Entry/Exit:
- VWAP = Σ(hl2 · volume) / Σ(volume), reset at each session boundary (vwap_session).
- Stdev = √(max(Σ(hl2² · volume)/Σ(volume) − VWAP², 0)).
- Entry band = vwap_band_devs[vwap_entry_band] × stdev around VWAP (default = 4.01σ).
- Long entry:  close_prev ≥ lower AND close_now < lower.
- Short entry: close_prev ≤ upper AND close_now > upper.
- Exit: close ≥ VWAP (long) or close ≤ VWAP (short) — first touch wins.

## Configuration

### Setup

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
import dataclasses

from engine.backtester import Backtester
from engine.data_configurator import ACTIVE, load_data, save_result, LIVE_DIR
from engine.strategy_configurator import params_for, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE
from engine.visualization import build_chart
from engine.evaluation import walk_forward, monte_carlo, grid_search, oracle_ceiling
from engine.live import LiveEngine

import pandas as pd
import plotly.express as px

### Automatic

In [ ]:
# Automatic config: project-wide defaults defined by the three configurators.
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = params_for("vwap_bands")   # engine/strategy_configurator.py (VwapParams — this notebook's family)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

### Manual


*_CONFIG = Automatic defaults, with any Manual overrides layered on top:
- Leave *_OVERRIDES empty → *_CONFIG is pure Automatic.
- Fill it → Automatic baseline + your Manual overrides.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic VwapParams().
# A foreign key raises TypeError here, not a silent no-op.
STRATEGY_OVERRIDES = {}      # e.g. {"vwap_entry_band": 3}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = each strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

### Final configuration

In [ ]:
# Prepare the final inputs the rest of the notebook uses.
# Runs after overrides.
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

# Report exactly what data + which configs are in force downstream (manual or automatic).
_window = (f"{DATA_CONFIG.start} → {DATA_CONFIG.end or 'now'}"
           if DATA_CONFIG.is_range else f"last {DATA_CONFIG.num_candles}")
tc = TRADING_CONFIG
_exits = (", ".join(f"{k!r}: {v!r}" for k, v in STRATEGY_CONFIG.EXITS.items())
          if EXIT_POLICY is None else f"override → {EXIT_POLICY}")
print(f"Loaded {len(df):,} candles | {SYMBOL} {INTERVAL}m {DATA_CONFIG.category} | "
      f"{_window} | {df.index[0]:%Y-%m-%d %H:%M} → {df.index[-1]:%Y-%m-%d %H:%M} UTC")
print(f"Trade: initial_equity={tc.initial_equity}, position_size_bps={tc.position_size_bps}, "
      f"leverage={tc.leverage}, sizing_mode={tc.sizing_mode.value!r}, "
      f"risk_per_trade_bps={tc.risk_per_trade_bps}, direction={tc.direction.value!r}")
print("Strategy Parameters: "
      + ", ".join(f"{k}={v}" for k, v in dataclasses.asdict(STRATEGY_CONFIG).items()))
print(f"Strategy exits: {_exits}")

<a id="vwap-bands"></a>
## VWAP Stdev Bands

<a id="vwap-bands-backtesting"></a>
### Backtesting

In [ ]:
# Import VWAP Stdev Bands strategy
from engine.strategies import VWAPBandsStrategy
STRATEGY = VWAPBandsStrategy

In [ ]:
# Backtest VWAP Stdev Bands strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="vwap_bands")

In [ ]:
# Theoretical ceiling
# For OHLCV-only crypto: ~5–15% is a realistic edge; under ~5% is noise.
_ceil_bps, _chain = oracle_ceiling(df, cost_bps=TRADING_CONFIG.total_cost_bps())
_skill = result.total_pnl_bps / _ceil_bps * 100 if _ceil_bps else 0.0
print(f"Theoretical ceiling : {_ceil_bps:+,.0f} bps  (full look-ahead, {TRADING_CONFIG.total_cost_bps():.0f} bps cost, {max(0, len(_chain) - 1)} trades)")
print(f"Strategy P&L   : {result.total_pnl_bps:+,.0f} bps")
print(f"Skill ratio    : {_skill:.1f}%  captured of what was theoretically possible")

In [ ]:
# VWAP Stdev Bands strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="vwap-bands-grid-search"></a>
### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"vwap_entry_band": [2, 3, 4]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

<a id="vwap-bands-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
# TRAIN_BARS is an in-sample window swept for the best params.
# TEST_BARS is an out-of-sample window the winner is then tested on.
# OBJECTIVE  can be any sweep metric: total_pnl_bps | sharpe_approx | profit_factor | ...
# MIN_TRADES lets ignore in-sample combos with fewer trades (noise, not signal)

GRID = {"vwap_entry_band": [2, 3, 4]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window.
# nunique==1 => the optimiser locked the same value every fold; wide min..max / large std => jumpy.
display(wf.param_stability())
wf.param_stability_summary()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
 .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits) over the strategy's base-config chart.
# Each fold re-optimises on its train window; see wf.folds_frame() for the per-fold params.
build_chart(strategy.prepare(df), trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | {SYMBOL} {INTERVAL}m").show()

<a id="vwap-bands-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

<a id="vwap-bands-live-signals"></a>
### Live signals

Live mode:
- runs the same strategy / config / costs as the backtest above
- generates signals: tells you when to enter / exit
- does not place orders
- the chart auto-refreshes every poll_seconds

Signal notification + sound:
- alerts you on each new entry/exit
- the first poll primes silently; alerts start from the next new signal
- browser = a banner + beep right in this cell's output (Safari/Chrome) \
  Runs via the notebook cell, not via CLI.
- desktop = a native macOS notification
- telegram reaches your phone by setting TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID in the environment

From the CLI:
- a terminal running the same loop
- prints a file link to the auto-refreshing chart
- add --notify to alert on each new signal: \
  python -m engine --strategy vwap_bands --mode live --interval 15 --poll 30 --notify desktop,telegram

From a notebook cell:
- prints a clickable chart link
- run() blocks the execution of the rest of the notebook until stopped

In [ ]:
live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
    notifiers="browser,desktop",        # ← edit here: browser / desktop / telegram, or None to disable
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button

<a id="vwap-analysis"></a>
## VWAP Analysis

<a id="band-hit-frequency"></a>
### Band hit frequency

In [ ]:
# How often each band is reached (intraday touch via high ≥ upper or low ≤ lower),
# plus how many long / short entries the strategy fires when that band is the trigger.
import pandas as pd
from dataclasses import replace

from engine.core import Direction, SignalAction

high = prepared["high"]
low = prepared["low"]
total = len(prepared)

# Per-band entry counts: re-run the strategy once with each band index as the trigger.
# Shorts come from the upper side, longs from the lower side.
entries_per_band: dict[int, tuple[int, int]] = {}
for k, _ in enumerate(STRATEGY_CONFIG.vwap_band_devs):
    cfg_k = replace(STRATEGY_CONFIG, vwap_entry_band=k)
    strat_k = VWAPBandsStrategy(cfg_k, exit_policy=EXIT_POLICY)
    result_k = Backtester(strat_k, symbol=SYMBOL, trading_config=TRADING_CONFIG).run(df, interval=INTERVAL)
    longs = sum(1 for t in result_k.trades if t.direction == Direction.LONG)
    shorts = sum(1 for t in result_k.trades if t.direction == Direction.SHORT)
    entries_per_band[k] = (longs, shorts)

rows = []
for k, mult in enumerate(STRATEGY_CONFIG.vwap_band_devs):
    longs, shorts = entries_per_band[k]
    upper_hits = int((high >= prepared[f"vwap_upper_{k}"]).sum())
    lower_hits = int((low  <= prepared[f"vwap_lower_{k}"]).sum())
    rows.append({"side": "upper", "band": k, "stdev": mult,
                 "hits": upper_hits, "pct_of_bars": upper_hits / total,
                 "long_entries": 0, "short_entries": shorts})
    rows.append({"side": "lower", "band": k, "stdev": mult,
                 "hits": lower_hits, "pct_of_bars": lower_hits / total,
                 "long_entries": longs, "short_entries": 0})

band_hits = (
    pd.DataFrame(rows)
      .sort_values("hits", ascending=False)
      .reset_index(drop=True)
)
band_hits["pct_of_bars"] = (band_hits["pct_of_bars"] * 100).round(2)
band_hits

<a id="tuning-the-entry-band"></a>
### Tuning the entry band

Lower vwap_entry_band to fire on closer bands and get more (but noisier) trades. \
Band 0 = 1.28σ (tightest), band 4 = 4.01σ (furthest, default). vwap_band_devs and \
vwap_session are also part of VwapParams if you want to retune the stdev \
ladder or use a non-daily anchor (e.g. "h" for hourly resets).

In [ ]:
from dataclasses import replace

# Example: fire on the 2.51σ band (index 2) instead of the furthest one
tight_config = replace(STRATEGY_CONFIG, vwap_entry_band=2)
tight_strategy = VWAPBandsStrategy(tight_config, exit_policy=EXIT_POLICY)
tight_result = Backtester(tight_strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG).run(df, interval=INTERVAL)
print(tight_result.summary())

<a id="entry-band-sweep"></a>
### Entry-band sweep

Re-run the strategy with vwap_entry_band set to each of the five band levels in turn and collect per-run profit metrics, sorted by total P&L (bps) descending. final_balance and return_pct compound a $100 starting balance through that run's trades.

In [ ]:
from dataclasses import replace
import pandas as pd

INITIAL_BALANCE = 100  # USD

sweep_rows = []
for k, mult in enumerate(STRATEGY_CONFIG.vwap_band_devs):
    cfg_k = replace(STRATEGY_CONFIG, vwap_entry_band=k)
    strat_k = VWAPBandsStrategy(cfg_k, exit_policy=EXIT_POLICY)
    r = Backtester(strat_k, symbol=SYMBOL, trading_config=TRADING_CONFIG).run(df, interval=INTERVAL)

    balance, peak, max_dd = INITIAL_BALANCE, INITIAL_BALANCE, 0.0
    for t in r.trades:
        balance *= (1 + t.pnl_bps / 10_000)
        peak = max(peak, balance)
        max_dd = max(max_dd, (peak - balance) / peak)

    sweep_rows.append({
        "entry_band": k,
        "stdev": mult,
        "trades": r.total_trades,
        "win_rate": r.win_rate,
        "total_pnl_bps": r.total_pnl_bps,
        "avg_pnl_bps": r.avg_pnl_bps,
        "profit_factor": r.profit_factor,
        "max_dd_bps": r.max_drawdown_bps,
        "final_balance": balance,
        "return_pct": (balance / INITIAL_BALANCE - 1) * 100,
    })

sweep = (
    pd.DataFrame(sweep_rows)
      .sort_values("total_pnl_bps", ascending=False)
      .reset_index(drop=True)
)
sweep["win_rate"] = (sweep["win_rate"] * 100).round(1)
sweep["total_pnl_bps"] = sweep["total_pnl_bps"].round(1)
sweep["avg_pnl_bps"] = sweep["avg_pnl_bps"].round(1)
sweep["profit_factor"] = sweep["profit_factor"].round(2)
sweep["max_dd_bps"] = sweep["max_dd_bps"].round(1)
sweep["final_balance"] = sweep["final_balance"].round(2)
sweep["return_pct"] = sweep["return_pct"].round(2)
sweep